<a href="https://colab.research.google.com/github/charlie-aashutosh/100-Next-Gen-Engg-C4/blob/main/LangChain_Workflows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-core langchain-community langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.0 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI #helpful for configuring the llm models from Gemini

In [3]:
import os #for creating virtual environments
import getpass #for secretly passing the API key

In [4]:
os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your API key")

Enter your API key··········


In [5]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash'
)

In [6]:
result=llm.invoke("What is the capital of India?")

In [7]:
print(result.content)

[{'type': 'text', 'text': 'The capital of India is **New Delhi**.', 'extras': {'signature': 'EpgFCpUFAWkUfRMqVobqVtu3huU6ooRrUfi8idK6SNtSxSC4qSkp15O0KoStQXaR02/3vXpIgfWvaAIXVINyHqkVWkhZkGLAo3p5RxmSydMLxivQ6UA8SquFP0Y2CpU1rwHeoMmu1j2Pn8qCWGg1MNRw89KjWaoK1Nkqp0UrnkLDR/lhMhAgrxIN1vQGV2yUMolomS15avWhnSkyTEb45JjCpUGInCHsXi33dIMhtm6Yz7m/foME2KoIZ5EvhqEK0NmC/2w8PYAw9lZjsQyP/WAk3dZ1R06QvrD9Ffm8Z2dBZojPRiJQs1vd+SGdp0C14Vgu+/tnQ4XXjdC7yQ8wDQYQUF/49dPfsgYNep6nRUCFyBaVPZa3npJepBKGdhcTYBwa8KzRULzB7elZHJ7ocwt/IzQE/NH35ee4Tkq2Q3KfbMcVzSkfINaPcplGDqXbB+sShL8VZxSD9E2n5y02VFpMmD/4T0o7pTYbB2qw+RpaZ8SmR8EvNSnP6SbrHQBmv4BJ/bYrqvlbhiLr2CQ98QaqOci5QskLSIV5xy7b5a14Tjb+qFlIQgP3U0TlDTWp1a769s5rH+/gpBQqHstYZ4Y+WguZjry+3gSlVFc+Fjk4SOWq5v1ipqxYnjiRYwscc4EbwoqAARTxE0hcCltiRu/vJaxvhT32YycXJvD7H4ckqrW+DA5+ZyMo8dAVsXl9jq0Qi7/WArVC0VIz8ecxDEFrYw++9FxJ84kknY8z4OemUXQRB1Jw7xZI/7NC8I9Jf821svWcdy6y5kUi4yYiBq+fYPKI9hm7U1pDfoEDNKNGjOwS6LyGBR8jier3gc0cgiUkQ5glF0j+KWsKl5qwYX2jhyMSbUNrYhrLCAS03pG+WFx4WQCohvIeUw=='}}]


Sequential Workflow using LangChain and a financial dataset

In [8]:
import pandas as pd
data=pd.read_excel("/content/My_Financial_Report_Dataset.xlsx")

In [9]:
data.head()

,Quarter,Business_Unit,Revenue_MUSD,COGS_MUSD,Operating_Expense_MUSD,EBITDA_MUSD,Net_Profit_MUSD,YoY_Growth_%,Market_Share_%,Customer_Churn_%,CapEx_MUSD,Region,Risk_Flag,Notes
0,Q1-2025,Payments,431,167.5,113.2,141.3,84.9,10.1,12.9,8.6,48.8,North America,Low,Cloud migration investment
1,Q2-2025,Corporate Banking,711,354.2,159.4,199.3,144.6,1.3,14.0,8.7,105.3,Europe,High,AI automation reduced servicing cost
2,Q1-2025,Wealth,319,178.9,68.8,67.7,7.7,15.0,12.4,6.0,93.9,Europe,Low,Higher regulatory compliance expense
3,Q2-2025,Payments,623,285.2,165.4,170.3,104.9,14.4,8.4,4.8,105.0,APAC,Medium,Cloud migration investment
4,Q2-2025,Wealth,235,93.7,55.1,93.2,45.1,-0.6,16.7,8.1,113.4,Europe,Low,Cloud migration investment


Chain 1 - is responsible for generating the Business KPIs by using the financial data as input

In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

In [11]:
#prompt for Chain 1
kpi_prompt=PromptTemplate(
    input_variables=["data"],
    template="""
    You are a Financial Analyst and you have been provided with a financial data of a Company.
    Your job is to extract all the meaningful financial and business KPIs from the data and
    present the output in form of bullet points

    financial data:{data}"""
)

In [12]:
#creating Chain 1
chain_1 = kpi_prompt | llm | StrOutputParser()

In [13]:
#executing Chain 1
kpis=chain_1.invoke({
    "data":data
})

In [14]:
print(kpis)

Here is an executive financial KPI dashboard and analysis based on the provided dataset for **FY2025**:

---

### **1. Executive Summary & Top-Line Financial KPIs**
* **Total Consolidated Revenue:** **$36,009.0 MUSD** (~$36.01 Billion)
* **Total Gross Profit:** **$19,439.2 MUSD** 
  * **Gross Profit Margin:** **53.98%** (Driven by revenue of $36.01B against COGS of $16.57B)
* **Total EBITDA:** **$9,625.5 MUSD**
  * **EBITDA Margin:** **26.73%**
* **Total Operating Expenses (OpEx):** **$8,843.1 MUSD** (24.56% of Total Revenue)
* **Total Net Profit:** **$6,568.1 MUSD**
  * **Net Profit Margin:** **18.24%**
* **Capital Expenditure (CapEx):** **$3,745.2 MUSD**
  * **CapEx Intensity (CapEx / Revenue):** **10.40%** (Reflects significant technology and infrastructure reinvestment)

---

### **2. Business Unit (BU) Performance & Metrics**
* **Corporate Banking (Core Revenue Driver):**
  * **Contribution:** Generated over **43%** of total corporate revenues.
  * **Peak Single Entry:** Recorded 

Chain 2 - is where we use the KPIs generated as output of Chain 1 as input for creating benchmarks

In [16]:
#prompt for Chain 2
benchmarks_prompt = PromptTemplate(
    input_variable=['kpis'],
    template="""
    You are a Senior Strategy Consultant and your job is to critically analyze the KPIs and
    establish appropriate performance benchmarks for management evaluation

    For all the major KPIs, provide the following:
    - Reasonable benchmarks/target
    - State the current performance
    - Classify the performance as:
      - Strong
      - Moderate
      - Needs improvement

    KPIs:{kpis}
    """
)

In [17]:
chain_2 = benchmarks_prompt | llm | StrOutputParser()

In [18]:
benchmarks = chain_2.invoke({
    "kpis":kpis
})

In [19]:
print(benchmarks)

### **Executive Strategic Evaluation Framework (FY2025)**

**To:** Executive Leadership & Board of Directors  
**From:** Senior Strategy Consultant  
**Subject:** FY2025 KPI Performance Benchmark & Management Evaluation  

---

### **Executive Summary**
The organization delivered solid top-line performance in FY2025 with **$36.01B in Total Revenue** and an **18.24% Net Profit Margin**, supported by strong core drivers like Corporate Banking and high-growth segments in Payments. 

However, critical operational efficiencies and risk exposures threaten sustainable value creation:
1. **Severe portfolio volatility:** Double-digit revenue drops in Insurance (-18.5%) and Corporate Banking (-5.7%) offset hyper-growth in Payments (+23.3%).
2. **Elevated cost structure:** Technology/compliance expenses push OpEx to **24.56%** of revenue, driving net loss quarters in Payments and Insurance despite high top-line scale.
3. **Customer Retention Risk:** Corporate Banking suffered peak customer churn 